In [40]:
from langchain_core.documents import Document

In [41]:
doc = Document(page_content="Hello, world!", metadata={"source": "example.txt","pages": 1,"author": "Yugank","date": "2026-06-03"})
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Yugank', 'date': '2026-06-03'}, page_content='Hello, world!')

In [42]:
## Create Directory for Text Files
import os
os.makedirs("../data/text_files", exist_ok=True)

In [43]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [44]:
## Textloader
### we can use the TextLoader class from langchain_community.document_loaders to load text files into Document objects. This allows us to easily work with the content of the text files in our applications.
from langchain_community.document_loaders import TextLoader
### we can use this also -> from langchain.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
documents = loader.load()
print(documents[0].page_content)

Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.


In [45]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader
### we can use this also -> from langchain.document_loaders import DirectoryLoader
loader = DirectoryLoader(
    "../data/text_files", ## directory path
    glob="**/*.txt", ## pattern to match files
    loader_cls=TextLoader, ### loader class to use for loading files
    loader_kwargs={"encoding": "utf-8"}, ## additional arguments to pass to the loader class
    show_progress=False
) 
documents = loader.load()
documents
print(f"✅ Loaded {len(documents)} documents from the directory.")
print(documents[0].page_content + "\n")
print(documents[1].page_content)

✅ Loaded 2 documents from the directory.
Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards 

In [46]:
### for pdf
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader,PyMuPDFLoader
### we can use this also -> from langchain.document_loaders import PyPDFLoader,PyMuPDFLoader
dir_loader = DirectoryLoader(
    "../data/pdf", ## directory path
    glob="**/*.pdf", ## pattern to match files
    loader_cls=PyMuPDFLoader, ### loader class to use for loading files
    show_progress=False
)
pdf_documents = dir_loader.load()
pdf_documents
print(f"✅ Loaded {len(pdf_documents)} PDF documents from the directory." + "\n")
print(pdf_documents[0].page_content + "\n")
print(pdf_documents[1].page_content)

✅ Loaded 2 PDF documents from the directory.

🇮🇳 Independence Day 
📅 Date 
15 August 
📌 What is it? 
Independence Day is celebrated every year to mark India's independence from British rule 
in 1947. 
⭐ Key Points 
●​ India became independent on 15 August 1947. 
●​ Jawaharlal Nehru became the first Prime Minister of independent India. 
●​ The Prime Minister hoists the National Flag at the Red Fort. 
●​ The day honors the sacrifices of freedom fighters. 
●​ It represents freedom, unity, and patriotism.

🇮🇳 Republic Day 
📅 Date 
26 January 
📌 What is it? 
Republic Day is celebrated every year to commemorate the adoption of the Constitution of 
India, which came into effect on 26 January 1950. 
⭐ Key Points 
●​ India became a republic on 26 January 1950. 
●​ The Constitution of India came into effect on this day. 
●​ The President of India unfurls the National Flag. 
●​ A grand Republic Day Parade is held in New Delhi. 
●​ It represents democracy, equality, and constitutional values.


In [47]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [48]:
### for CSV
from langchain_community.document_loaders import CSVLoader
### we can use this also -> from langchain.document_loaders import CSVLoader
csv_loader = CSVLoader(
    file_path="../data/csv/sales.csv", ## path to the CSV file
    encoding="utf-8", ## encoding of the CSV file
    csv_args={"delimiter": ","}, ## additional arguments to pass to the CSV reader
)
csv_documents = csv_loader.load()
csv_documents
print(f"✅ Loaded {len(csv_documents)} CSV documents from the directory." + "\n")
print(csv_documents[0].page_content + "\n")
print(csv_documents[1].page_content)

✅ Loaded 10 CSV documents from the directory.

order_id: 1001
product: Laptop
category: Electronics
quantity: 2
price: 65000
city: Delhi
order_date: 2026-01-05

order_id: 1002
product: Mouse
category: Accessories
quantity: 5
price: 800
city: Mumbai
order_date: 2026-01-07


In [49]:
type(csv_documents[0])

langchain_core.documents.base.Document

In [50]:
import os

from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path

In [51]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = [] #initialize an empty list to store all documents
    pdf_dir = Path(pdf_directory) # create a Path object for the PDF directory
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files: # iterate through each PDF file
        print(f"\nProcessing: {pdf_file.name}") # print the name of the PDF file being processed
        try: # load the PDF file using PyPDFLoader
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load() # load the PDF file and return a list of Document objects
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name # add the name of the PDF file to the metadata
                doc.metadata['file_type'] = 'pdf' # add the file type to the metadata
            
            all_documents.extend(documents) # add the loaded documents to the all_documents list
            print(f"  ✓ Loaded {len(documents)} pages") # print the number of pages loaded from the PDF file
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}") # print the total number of documents loaded
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Independence.pdf
  ✓ Loaded 1 pages

Processing: Republic.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [52]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../data/pdf/Independence.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Independence.pdf', 'file_type': 'pdf'}, page_content="\U000f03dd\n \nIndependence\n \nDay\n \n📅\n \nDate\n \n15\n \nAugust\n \n📌\n \nWhat\n \nis\n \nit?\n \nIndependence\n \nDay\n \nis\n \ncelebrated\n \nevery\n \nyear\n \nto\n \nmark\n \nIndia's\n \nindependence\n \nfrom\n \nBritish\n \nrule\n \nin\n \n1947\n.\n \n⭐\n \nKey\n \nPoints\n \n●\n \nIndia\n \nbecame\n \nindependent\n \non\n \n15\n \nAugust\n \n1947\n.\n \n●\n \nJawaharlal\n \nNehru\n \nbecame\n \nthe\n \nfirst\n \nPrime\n \nMinister\n \nof\n \nindependent\n \nIndia.\n \n●\n \nThe\n \nPrime\n \nMinister\n \nhoists\n \nthe\n \nNational\n \nFlag\n \nat\n \nthe\n \nRed\n \nFort\n.\n \n●\n \nThe\n \nday\n \nhonors\n \nthe\n \nsacrifices\n \nof\n \nfreedom\n \nfighters\n.\n \n●\n \nIt\n 

In [53]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200): # Split documents into smaller chunks for better RAG performance
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(     # Initialize the text splitter
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)   #split the documents into smaller chunks using the text splitter
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [54]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)
chunks

Split 2 documents into 2 chunks

Example chunk:
Content: 󰏝
 
Independence
 
Day
 
📅
 
Date
 
15
 
August
 
📌
 
What
 
is
 
it?
 
Independence
 
Day
 
is
 
celebrated
 
every
 
year
 
to
 
mark
 
India's
 
independence
 
from
 
British
 
rule
 
in
 
1947
.
 ...
Metadata: {'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../data/pdf/Independence.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Independence.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../data/pdf/Independence.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Independence.pdf', 'file_type': 'pdf'}, page_content="\U000f03dd\n \nIndependence\n \nDay\n \n📅\n \nDate\n \n15\n \nAugust\n \n📌\n \nWhat\n \nis\n \nit?\n \nIndependence\n \nDay\n \nis\n \ncelebrated\n \nevery\n \nyear\n \nto\n \nmark\n \nIndia's\n \nindependence\n \nfrom\n \nBritish\n \nrule\n \nin\n \n1947\n.\n \n⭐\n \nKey\n \nPoints\n \n●\n \nIndia\n \nbecame\n \nindependent\n \non\n \n15\n \nAugust\n \n1947\n.\n \n●\n \nJawaharlal\n \nNehru\n \nbecame\n \nthe\n \nfirst\n \nPrime\n \nMinister\n \nof\n \nindependent\n \nIndia.\n \n●\n \nThe\n \nPrime\n \nMinister\n \nhoists\n \nthe\n \nNational\n \nFlag\n \nat\n \nthe\n \nRed\n \nFort\n.\n \n●\n \nThe\n \nday\n \nhonors\n \nthe\n \nsacrifices\n \nof\n \nfreedom\n \nfighters\n.\n \n●\n \nIt\n 

EMBEDDING AND VECTOR STORE DB

In [55]:
import numpy as np 
from sentence_transformers import SentenceTransformer ## embedding model
import chromadb 
from chromadb.config import Settings 
import uuid ## for generating unique identifiers , because we will use it to generate unique IDs for our documents in the vector store 
from typing import List, Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity ## for calculating cosine similarity between vectors,while doing retrieval from vector db

In [56]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""    ### constructors are special methods that are called when an object is created from a class and allow the class to initialize the attributes of the class.
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):  ## this is for the model name of the sentence transformer model we want to use for generating embeddings
        """Initializes the EmbeddingManager with a specified model.
        Args:
            model_name (str): Hugging Face model name for the SentenceTransformer.
        """
        self.model_name = model_name ## intilialize the model name attribute with the provided model name
        self.model = None ## we intialize the model attribute to None, which will later hold the actual model instance
        self._load_model() ## Load the model during initialization
    
    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            print(f'Loading embedding model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name) ## we load the model using the SentenceTransformer class and store it in the model attribute
            print(f"Model loaded successfully. Embedding dimensions : {self.model.get_embedding_dimension()}") ## 384 default
        except Exception as e:
            print(f"Error loading model: {self.model_name}. Exception: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        Args:
            texts (List[str]): List of text strings to embed.
        Returns:
            np.ndarray: Array of embedding vectors.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Please check the model name or loading process.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True) ## we generate embeddings for text 
        print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self) -> int:
    #     """Returns the dimension of the embeddings produced by the model."""
    #     if not self.model:
    #         raise ValueError("Model is not loaded. Please check the model name or loading process.")
    #     return self.model.get_embedding_dimension()
    
    ### initialize the embedding manager
    
embedding_manager = EmbeddingManager()  ## we create an instance of the EmbeddingManager class, specifying the model name to use for generating embeddings 
embedding_manager
    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9906.52it/s]


Model loaded successfully. Embedding dimensions : 384


In [57]:
texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

print("Embedding shape:", embeddings.shape)
print(embeddings[0].shape)
embeddings.shape          # ✅ Correct
embeddings[0].shape

Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.17it/s]

Embeddings generated successfully. Shape: (2, 384)
Embedding shape: (2, 384)
(384,)


(384,)

In [58]:
texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

print(embeddings.shape)

Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 68.80it/s]

Embeddings generated successfully. Shape: (2, 384)
(2, 384)


VECTORE STORE

In [59]:
import os

class VectorStore:
    """Handles storage and retrieval of document embeddings using ChromaDB."""
    def __init__(self, collection_name: str = "pdf_document", persist_directory: str = "../data/vector_store"):
        """Initializes the VectorStore with a specified collection name and persistence directory.
        Args:
            collection_name (str): Name of the ChromaDB collection.
            persist_directory (str): Directory to persist the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory) ## create a client for the ChromsDB
            
            #Get or create the collection. && where the vectors areto stored
            self.collection = self.client.get_or_create_collection(name=self.collection_name , metadata={"description":"PDF document embeddings for RAG"})
            print(f"Vector store initialized successfully. Collection: {self.collection_name}, Persist directory: {self.persist_directory}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store. Exception: {e}")
            raise
        
    ##for adding documents to the vector store
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Adds documents to the vector store after generating embeddings.
        Args:
            documents : List of LangChain Documents
            embedding (np.ndarray): Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        
        #preapre data for chromaDb
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []
        
        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            #Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  ## we generate a unique ID for each document using the uuid library, which creates a random UUID and we take the first 8 characters of its hexadecimal representation
            ids.append(doc_id)
            
            #Prepare metadata
            metadata = dict(doc.metadata)  ## we create a copy of the document's metadata to avoid modifying the original metadata
            metadata['doc_index'] = i  ## we add an additional key-value pair to the metadata dictionary, indicating the index of the document in the list
            metadata['content_length'] = len(doc.page_content)  ## we add another key-value pair to the metadata dictionary, indicating the length of the document's content
            metadatas.append(metadata)
            
            #Documents content
            documents_texts.append(doc.page_content)
            
            #Embeddings
            embeddings_list.append(embedding.tolist())  ## we convert the embedding from a numpy array to a list, as ChromaDB expects embeddings in list format
        
        #Add to ChromaDB collection    
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store. Exception: {e}")
            raise
            
        
vector_store = VectorStore()
vector_store.add_documents(chunks, embeddings)

Vector store initialized successfully. Collection: pdf_document, Persist directory: ../data/vector_store
Existing documents in collection: 10
Adding 2 documents to the vector store...
Successfully added 2 documents to the vector store.
Documents added successfully. Total documents in collection: 12


In [60]:
chunks

[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../data/pdf/Independence.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Independence.pdf', 'file_type': 'pdf'}, page_content="\U000f03dd\n \nIndependence\n \nDay\n \n📅\n \nDate\n \n15\n \nAugust\n \n📌\n \nWhat\n \nis\n \nit?\n \nIndependence\n \nDay\n \nis\n \ncelebrated\n \nevery\n \nyear\n \nto\n \nmark\n \nIndia's\n \nindependence\n \nfrom\n \nBritish\n \nrule\n \nin\n \n1947\n.\n \n⭐\n \nKey\n \nPoints\n \n●\n \nIndia\n \nbecame\n \nindependent\n \non\n \n15\n \nAugust\n \n1947\n.\n \n●\n \nJawaharlal\n \nNehru\n \nbecame\n \nthe\n \nfirst\n \nPrime\n \nMinister\n \nof\n \nindependent\n \nIndia.\n \n●\n \nThe\n \nPrime\n \nMinister\n \nhoists\n \nthe\n \nNational\n \nFlag\n \nat\n \nthe\n \nRed\n \nFort\n.\n \n●\n \nThe\n \nday\n \nhonors\n \nthe\n \nsacrifices\n \nof\n \nfreedom\n \nfighters\n.\n \n●\n \nIt\n 

In [61]:
# Get text from chunks
texts = [chunk.page_content for chunk in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Check shape
print(embeddings.shape)

# Add to vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 72.48it/s]

Embeddings generated successfully. Shape: (2, 384)
(2, 384)
Adding 2 documents to the vector store...
Successfully added 2 documents to the vector store.
Documents added successfully. Total documents in collection: 14


RAG Retriever Pipeline From Vector Space

In [62]:
class RAGRetriever:
    """Handles retrieval of relevant documents from the vector store based on a query."""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initializes the RAGRetriever with a vector store and embedding manager.
        Args:
            vector_store (VectorStore): Instance of the VectorStore class.
            embedding_manager (EmbeddingManager): Instance of the EmbeddingManager class.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    #Retriever based on specific query.... TOP k is used to get the top k results based on similarity score, 
    # int = number of results to return
    # score_threshold is used to filter out results below a certain similarity 
    # float = minimum similarity score to consider a document relevant
    def retrieve(self, query: str, top_k: int = 5,score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieves relevant documents for a given query.
        Args:
            query (str): The input query string.
            top_k (int): Number of top results to return.
            score_threshold (float): Minimum similarity score to consider a document relevant.
        Returns:
            List[Dict[str, Any]]: List of retrieved documents with their metadata and similarity scores.
        """
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
        
        #Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])  ## query need to be converted to the embedding
        
        #Search in the vector store
        try:
            results = self.vector_store.collection.query(query_embeddings = query_embedding.tolist(),n_results=top_k)  ## we search the vector store for the top_k most similar documents to the query 
            #Process results
            retrieved_docs = []
            
            # internally there will be the key "documents" in the results dictionary, which contains the list of retrieved documents.
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]  ## we access the first element of the "documents" list, which contains the actual retrieved documents
                metadatas = results['metadatas'][0]  ## we access the first element of the "metadatas" list, which contains the metadata for the retrieved documents
                distances = results['distances'][0]  ## we access the first element of the "distances" list, which contains the similarity scores for the retrieved documents
                ids = results['ids'][0]  ## we access the first element of the "ids" list, which contains the unique IDs for the retrieved documents
                
                # for all the parametere, ziping (creating a tuyple) and iterating through them, so that we can access the document, metadata, distance and id of each retrieved document in a single iteration
                for i,(documents,metadata,distance,doc_id) in enumerate(zip(documents,metadatas,distances,ids)):
                    ## we calculate the similarity score by subtracting the distance from 1, as a lower distance indicates higher similarity
                    similarity_score = 1 - distance  
                    ## we check if the similarity score of the document is above the specified threshold
                    if similarity_score >= score_threshold:  
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": documents,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1  ## we assign a rank to the document based on its position in the list of retrieved documents, starting from 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold.")
            else:
                print("No documents retrieved.")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval. Exception: {e}")
            raise

rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [63]:
# it will return and object of RAGRetriever class.
rag_retriever

In [64]:
rag_retriever.retrieve("When we celebrate the Republic Day of India?")

Retrieving documents for query: 'When we celebrate the Republic Day of India?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 149.52it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 5 documents after applying score threshold.


[{'id': 'doc_8b09dcf4_1',
  'content': '\U000f03dd\n \nRepublic\n \nDay\n \n📅\n \nDate\n \n26\n \nJanuary\n \n📌\n \nWhat\n \nis\n \nit?\n \nRepublic\n \nDay\n \nis\n \ncelebrated\n \nevery\n \nyear\n \nto\n \ncommemorate\n \nthe\n \nadoption\n \nof\n \nthe\n \nConstitution\n \nof\n \nIndia\n,\n \nwhich\n \ncame\n \ninto\n \neffect\n \non\n \n26\n \nJanuary\n \n1950\n.\n \n⭐\n \nKey\n \nPoints\n \n●\n \nIndia\n \nbecame\n \na\n \nrepublic\n \non\n \n26\n \nJanuary\n \n1950.\n \n●\n \nThe\n \nConstitution\n \nof\n \nIndia\n \ncame\n \ninto\n \neffect\n \non\n \nthis\n \nday.\n \n●\n \nThe\n \nPresident\n \nof\n \nIndia\n \nunfurls\n \nthe\n \nNational\n \nFlag.\n \n●\n \nA\n \ngrand\n \nRepublic\n \nDay\n \nParade\n \nis\n \nheld\n \nin\n \nNew\n \nDelhi.\n \n●\n \nIt\n \nrepresents\n \ndemocracy,\n \nequality,\n \nand\n \nconstitutional\n \nvalues\n.',
  'metadata': {'doc_index': 1,
   'page': 0,
   'source_file': 'Republic.pdf',
   'creator': 'PyPDF',
   'file_type': 'pdf',
   'content

Integration Vectoredb Context pipeline With LLM Output

In [65]:
### Simple RAG pipeline with GroQ LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize GroQ API key from environment variable
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize the GroQ LLM Model
llm = ChatGroq(api_key=groq_api_key, model="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

## Create a simple RAG pipeline that takes a query, retrieves relevant documents, and generates a response using the GroQ LLM.
def rag_simple(query,retriever,llm,top_k=3):
    ## Retrieve the context
    result = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in result]) if result else "" # if result is not empty, join the content of the retrieved documents with double newlines, otherwise set context to an empty string
    print(f"Retrieved context:\n{context}\n")
    
    if not context:
        return "No relevant documents found for the query."
    
    # Generate response using the GroQ 
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    # we invoke the GroQ LLM with the prompt, which includes the retrieved context and the user's query, and we return the content of the first response generated by the model.
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content ## we return the content of the first response generated

In [66]:
answer = rag_simple("When we celebrate the Republic Day of India?", rag_retriever, llm, top_k=3)
print(answer)

Retrieving documents for query: 'When we celebrate the Republic Day of India?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 147.71it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents after applying score threshold.
Retrieved context:
󰏝
 
Republic
 
Day
 
📅
 
Date
 
26
 
January
 
📌
 
What
 
is
 
it?
 
Republic
 
Day
 
is
 
celebrated
 
every
 
year
 
to
 
commemorate
 
the
 
adoption
 
of
 
the
 
Constitution
 
of
 
India
,
 
which
 
came
 
into
 
effect
 
on
 
26
 
January
 
1950
.
 
⭐
 
Key
 
Points
 
●
 
India
 
became
 
a
 
republic
 
on
 
26
 
January
 
1950.
 
●
 
The
 
Constitution
 
of
 
India
 
came
 
into
 
effect
 
on
 
this
 
day.
 
●
 
The
 
President
 
of
 
India
 
unfurls
 
the
 
National
 
Flag.
 
●
 
A
 
grand
 
Republic
 
Day
 
Parade
 
is
 
held
 
in
 
New
 
Delhi.
 
●
 
It
 
represents
 
democracy,
 
equality,
 
and
 
constitutional
 
values
.

󰏝
 
Republic
 
Day
 
📅
 
Date
 
26
 
January
 
📌
 
What
 
is
 
it?
 
Republic
 
Day
 
is
 
celebrated
 
every
 
year
 
to
 
commemorate
 
the
 
adoption
 
of
 
the
 
Constitution
 
of
 
India
,
 
which
 
came
 
into
 
effect
 
on
 
2

India celebrates Republic Day every year on **26 January**.


Enhanced RAG pipeline features

In [67]:
# Enhanced RAG pipeline Features
def rag_advanced(query, retriever, llm, top_k=3,min_score=0.2, return_context=False):
    """RAG pipeline with extra features: return answers,sources,confidence score , and optionally full context."""
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score) # Retrieve relevant documents
    if not results:
        return {'answer': "No relevant documents found for the query.", 'sources': [], 'confidence': 0.0, 'context': ""}
    
    #Prepare context for the LLM
    context = "\n\n".join([doc['content'] for doc in results])
    # we create sources in which we extract the source file name, page number, similarity score, and a preview of the content for each retrieved document.
    # This information is useful for providing transparency and traceability in the RAG process.
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
        'page' : doc['metadata'].get('page','unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:300] + '...'
    } for doc in results] # we do this by iterating through the retrieved documents and extracting the relevant metadata and content for each document.
    confidence = max([doc['similarity_score'] for doc in results])  ## we calculate the confidence score as the maximum similarity score among the retrieved documents
    
    #Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion: {query}\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    # output formate
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    # if return_context is True, we include the full context in the output dictionary, allowing users to see the entire retrieved context along with the answer and sources.
    if return_context:
        output['context'] = context
    return output
    
result = rag_advanced("When we celebrate the Republic Day of India?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer : " + result['answer'] + "\n")
print("Sources : " + str(result['sources']) + "\n")
print("Confidence : " + str(result['confidence']) + "\n")
print("Context Preview : " + result['context'][:300] + "\n")
    

Retrieving documents for query: 'When we celebrate the Republic Day of India?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 133.73it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents after applying score threshold.


Answer : India’s Republic Day is celebrated every year on 26 January.

Sources : [{'source': 'Republic.pdf', 'page': 0, 'score': 0.3916736841201782, 'preview': '\U000f03dd\n \nRepublic\n \nDay\n \n📅\n \nDate\n \n26\n \nJanuary\n \n📌\n \nWhat\n \nis\n \nit?\n \nRepublic\n \nDay\n \nis\n \ncelebrated\n \nevery\n \nyear\n \nto\n \ncommemorate\n \nthe\n \nadoption\n \nof\n \nthe\n \nConstitution\n \nof\n \nIndia\n,\n \nwhich\n \ncame\n \ninto\n \neffect\n \non\n \n26\n \nJanuary\n \n1950\n.\n \n⭐\n \nKey\n \nPoints\n \n●\n \nIndia\n \nbecame\n \na\n \n...'}, {'source': 'Republic.pdf', 'page': 0, 'score': 0.3916736841201782, 'preview': '\U000f03dd\n \nRepublic\n \nDay\n \n📅\n \nDate\n \n26\n \nJanuary\n \n📌\n \nWhat\n \nis\n \nit?\n \nRepublic\n \nDay\n \nis\n \ncelebrated\n \nevery\n \nyear\n \nto\n \ncommemorate\n \nthe\n \nadoption\n \nof\n \nthe\n \nConstitution\n \nof\n \nIndia\n,\n \nwhich\n \ncame\n \ninto\n \neffect\n \non\n \n26\n \nJanuary\n \n1950\n.\n \n⭐\n \nKey\n \nPoints\n \n

ADVANCED RAG PIPELINE

In [68]:
## ADVANCED RAG PIPELINE with streaming , citation, history,summarization.
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    # intialize the class with retriever and llm
    def __init__(self, retriever, llm):
        self.retriever = retriever # reteiver instance for document retrieval
        self.llm = llm # llm instance for generating responses
        self.history = []  # Store query history

    # Query method to handle user queries with advanced features
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            # if stream is True, we simulate streaming the answer by printing the prompt in chunks of 80 characters with a short delay between each chunk. This gives the user the impression that the answer is being generated in real-time.
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to  the answer  --> citation means to provide references to the sources of information used in the answer, allowing users to verify the information and explore the original documents for more details.
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)] 
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Why do we celebrate Republic Day?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Why do we celebrate Republic Day?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 119.00it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents after applying score threshold.
Streaming answer:
Use the following context to answer the question concisely.
Context:
󰏝
 
Republic
 
Day
 
📅
 
Date
 
26
 
January
 
📌
 
What
 
is
 
it?
 
Republic
 
Day
 
is
 
celebrated
 
every
 
year
 
to
 
commemorate
 
the
 
adoption
 
of
 
the
 
Constitution
 
of
 
India
,
 
which
 
came
 
into
 
effect
 
on
 
26
 
January
 
1950


.
 
⭐
 
Key
 
Points
 
●
 
India
 
became
 
a
 
republic
 
on
 
26
 
January
 
1950.
 
●
 
The
 
Constitution
 
of
 
India
 
came
 
into
 
effect
 
on
 
this
 
day.
 
●
 
The
 
President
 
of
 
India
 
unfurls
 
the
 
National
 
Flag.
 
●
 
A
 
grand
 
Republic
 
Day
 
Parade
 
is
 
held
 
in
 
New
 
Delhi.
 
●
 
It
 
represents
 
democracy,
 
equality,
 
and
 
constitutional
 
values
.

󰏝
 
Republic
 
Day
 
📅
 
Date
 
26
 
January
 
📌
 
What
 
is
 
it?
 
Republic
 
Day
 
is
 
celebrated
 
every
 
year
 
to
 
commemorate
 
the
 
adoption
 
of
 
the
 
Constitution
 
of
 
India
,
 
which
 
came
 
into
 
effect
 
on
 
26
 
January
 
1950
.
 
⭐
 
Key
 
Points
 
●
 
India
 
became
 
a
 
republic
 
on
 
26
 
January
 
1950.
 
●
 
The
 
Constitution
 
of
 
India
 
came
 
into
 
effect
 
on
 
this
 
day.
 
●
 
The
 
President
 
of
 
India
 
unfurls
 
the
 
National
 
Flag.
 
●
 
A
 
grand
 
Republic
 
Day
 
Parade
 
is
 
held
 
in
 
New
 
Delhi.
 
●
 
It
 
represents
 
democracy,
 
equality,
